# CelebA-HQ — read-only path index for the mounted DIRE tree

Attach the existing full DIRE Kaggle Dataset as input, then run the definition cell and the index cell. Keep Accelerator Off. The index scans only `train/celebahq` and the selected `test/celebahq` groups.

It does not modify `/kaggle/input`, copy data, open/decode images, compute hashes, create validation splits, or train. It writes one JSON index under `/kaggle/working`. The member-style locator is inferred from the extracted tree and is explicitly not verified against original tar archives.

Stop after reviewing the counts and locator fields. Do not run a training notebook from this path index alone.

## Stage 2 — SHA-256 + full image decode (opt-in)

Chỉ chạy stage này sau khi path-index hoàn tất. Nó đọc từng file hai lượt (SHA-256, sau đó giải mã đầy đủ bằng Pillow), nhưng không ghi vào input và không dùng GPU. Checkpoint SQLite được lưu ngoài input tại `/kaggle/working`; nếu cell bị ngắt trong cùng runtime, chạy lại để resume. Trước khi kết thúc phiên, hãy lưu/export checkpoint và report nếu cần tiếp tục ở phiên khác.

Locator vẫn là suy luận từ cây đã giải nén; SHA-256 xác định bytes hiện tại, không xác nhận archive gốc/provenance.


In [ ]:
"""Read-only inventory adapter for the extracted CelebA-HQ DIRE maps on Kaggle.

The Kaggle input reported by the user is an extracted directory tree, e.g.
``dire/train/celebahq/real/real/<image>``. This module resolves the repeated
leaf folder without copying or changing the mounted data. The inferred member
locator is explicitly not proof of the original tar member path; archive bytes
and provenance are unavailable from this extracted tree.

This is an inventory/path adapter only. It does not assign validation splits,
open/decode images, compute hashes, train, or modify input data.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

IMAGE_SUFFIXES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
REQUIRED_GROUPS: tuple[tuple[str, str], ...] = (
    ("train", "real"),
    ("train", "sdv2"),
    ("test", "real"),
    ("test", "dalle2"),
    ("test", "if"),
    ("test", "midjourney"),
    ("test", "stargan"),
    ("test", "sdv2"),
)


class CelebAHQTreeLayoutError(ValueError):
    """The extracted tree is missing a required group or has ambiguous layout."""


@dataclass(frozen=True)
class CelebAHQImage:
    """One image path and its class metadata; no image bytes are read."""

    split: str
    generator: str
    label: int
    path: Path
    relative_path: str
    member_locator: str
    locator_inferred: bool
    leaf_layout: str


def _iter_image_files(directory: Path):
    """Yield regular image files in stable order without following symlinks."""
    for current, dirnames, filenames in os.walk(directory, topdown=True, followlinks=False):
        current_path = Path(current)
        dirnames[:] = sorted(
            name for name in dirnames
            if not (current_path / name).is_symlink()
        )
        for filename in sorted(filenames):
            path = current_path / filename
            if path.is_symlink() or path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            if path.is_file():
                yield path


def _direct_images(directory: Path) -> list[Path]:
    return sorted(
        path for path in directory.iterdir()
        if path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_SUFFIXES
    )


def scan_celebahq_tree(root: str | Path) -> list[CelebAHQImage]:
    """Index the selected CelebA-HQ train/test classes from an extracted tree.

    ``root`` is the mounted ``dire`` directory. For each generator folder, the
    scanner prefers its repeated same-name child (``real/real``) when it holds
    images and the outer folder has no direct images. If images exist both
    directly and below the repeated child, it raises instead of silently
    choosing one. A flat ``real/<image>`` layout is also accepted.

    ``member_locator`` approximates a full member-style locator such as
    ``real/0001.png`` by prefixing the generator name to the path under the
    selected leaf. ``locator_inferred`` remains true because this dataset is
    extracted and the original tar archives were not verified in this scan.
    """
    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"DIRE root is not a directory: {root_path}")

    records: list[CelebAHQImage] = []
    missing: list[str] = []
    for split, generator in REQUIRED_GROUPS:
        outer = root_path / split / "celebahq" / generator
        if not outer.is_dir():
            missing.append(f"{split}/celebahq/{generator}")
            continue

        direct = _direct_images(outer)
        repeated = outer / generator
        repeated_images = (
            list(_iter_image_files(repeated))
            if repeated.is_dir() and not repeated.is_symlink()
            else []
        )
        other_nested_images = []
        for child in sorted(outer.iterdir()):
            if child == repeated or not child.is_dir() or child.is_symlink():
                continue
            other_nested_images.extend(_iter_image_files(child))
        if other_nested_images:
            raise CelebAHQTreeLayoutError(
                f"unexpected nested image directories under {outer}: "
                f"{[path.relative_to(outer).as_posix() for path in other_nested_images[:5]]}"
            )
        if direct and repeated_images:
            raise CelebAHQTreeLayoutError(
                f"ambiguous class directory {outer}: found direct images and images under {repeated}"
            )
        if repeated_images:
            leaf_root = repeated
            leaf_layout = "repeated-generator-child"
            files = repeated_images
        elif direct:
            leaf_root = outer
            leaf_layout = "flat-class-directory"
            files = direct
        else:
            missing.append(f"{split}/celebahq/{generator} (no image files in expected leaf)")
            continue

        for path in files:
            relative = path.relative_to(root_path).as_posix()
            member_inside_leaf = path.relative_to(leaf_root).as_posix()
            member_locator = (Path(generator) / member_inside_leaf).as_posix()
            records.append(CelebAHQImage(
                split=split,
                generator=generator,
                label=0 if generator == "real" else 1,
                path=path,
                relative_path=relative,
                member_locator=member_locator,
                locator_inferred=True,
                leaf_layout=leaf_layout,
            ))

    if missing:
        raise CelebAHQTreeLayoutError(
            "missing or empty required CelebA-HQ groups: " + ", ".join(missing)
        )
    return records


In [ ]:
# Read-only path index: only the selected CelebA-HQ branches, no GPU/data copy
import json
from collections import Counter
from pathlib import Path

# Exact mounted DIRE root from the submitted Kaggle audit; edit only if Kaggle's mount path changes.
DIRE_ROOT = Path('/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire')
REPORT_PATH = Path('/kaggle/working/celebahq-path-index.json')

records = scan_celebahq_tree(DIRE_ROOT)
counts = Counter((row.split, row.generator) for row in records)
report = {
    'schema_version': 1,
    'status': 'complete',
    'root': str(DIRE_ROOT),
    'purpose': 'CelebA-HQ path index only; no pixel read/decode, hash, copy, or training',
    'input_modified': False,
    'image_contents_read': False,
    'gpu_work_started': False,
    'archive_member_paths_verified': False,
    'locator_policy': 'inferred by prefixing generator to path relative to selected leaf directory',
    'validation_split_assigned': False,
    'counts': [
        {'split': split, 'generator': generator, 'label': 0 if generator == 'real' else 1, 'files': n}
        for (split, generator), n in sorted(counts.items())
    ],
    'records': [
        {
            'split': row.split,
            'generator': row.generator,
            'label': row.label,
            'path_from_dire_root': row.relative_path,
            'member_locator_inferred': row.member_locator,
            'locator_inferred': row.locator_inferred,
            'leaf_layout': row.leaf_layout,
        }
        for row in records
    ],
    'notes': [
        'Counts are regular image files recognized by suffix; images were not opened or decoded.',
        'member_locator_inferred is a stable path inference, not a verified original tar member path or source identity.',
        'Only train/test CelebA-HQ groups are indexed; LSUN and ImageNet are left untouched.',
    ],
}
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Index status:', report['status'])
print('Indexed files:', len(records))
for row in report['counts']:
    print(f"  {row['split']:5s} {row['generator']:12s} label={row['label']} n={row['files']}")
print('Manifest:', REPORT_PATH)


In [ ]:
"""Read-only SHA-256 and image-decode audit for the selected CelebA-HQ tree.

The mounted input is never changed. Checkpoint database and exported manifests
are written only beneath a caller-provided writable directory outside the input.
The SQLite checkpoint can resume completed rows for the same file-tree metadata
fingerprint (root-relative paths, sizes, and mtimes).
"""

from __future__ import annotations

import hashlib
import json
import os
import sqlite3
import time
from pathlib import Path
from typing import Any, Iterable

CHUNK_SIZE = 1024 * 1024


def _inside(path: Path, root: Path) -> bool:
    return path == root or root in path.parents


def _source_fingerprint(records: list[Any], root: Path) -> str:
    digest = hashlib.sha256()
    for record in sorted(records, key=lambda row: row.relative_path):
        path = Path(record.path).expanduser().resolve()
        if not _inside(path, root):
            raise ValueError(f"indexed input path escapes DIRE root: {record.relative_path}")
        stat = path.stat()
        for value in (
            record.relative_path,
            record.split,
            record.generator,
            str(int(record.label)),
            record.member_locator,
            str(stat.st_size),
            str(stat.st_mtime_ns),
        ):
            digest.update(value.encode("utf-8"))
            digest.update(b"\0")
        digest.update(b"\n")
    digest.update(str(root).encode("utf-8"))
    return digest.hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(CHUNK_SIZE), b""):
            digest.update(block)
    return digest.hexdigest()


def _init_db(db: sqlite3.Connection, *, root: Path, fingerprint: str, count: int) -> None:
    db.execute("CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT NOT NULL)")
    db.execute(
        """CREATE TABLE IF NOT EXISTS files (
            relative_path TEXT PRIMARY KEY,
            split TEXT NOT NULL,
            generator TEXT NOT NULL,
            label INTEGER NOT NULL,
            member_locator TEXT NOT NULL,
            locator_inferred INTEGER NOT NULL,
            size_bytes INTEGER NOT NULL,
            mtime_ns INTEGER NOT NULL,
            sha256 TEXT,
            hash_status TEXT NOT NULL,
            decode_status TEXT NOT NULL,
            image_format TEXT,
            width INTEGER,
            height INTEGER,
            mode TEXT,
            error TEXT
        )"""
    )
    expected = {
        "schema_version": "1",
        "root": str(root),
        "source_fingerprint": fingerprint,
        "record_count": str(count),
    }
    existing = dict(db.execute("SELECT key, value FROM meta"))
    if existing and existing != expected:
        raise ValueError("checkpoint metadata mismatch; refusing to reuse another input manifest")
    if not existing:
        db.executemany("INSERT INTO meta(key, value) VALUES (?, ?)", expected.items())


def verify_celebahq_records(
    root: str | Path,
    records: Iterable[Any],
    workdir: str | Path,
    *,
    checkpoint_every: int = 250,
    progress_every: int = 1000,
) -> dict[str, Any]:
    """Hash and fully decode each indexed file, with resumable external checkpoint.

    Records are expected to expose ``split``, ``generator``, ``label``, ``path``,
    ``relative_path``, ``member_locator``, and ``locator_inferred`` (as emitted by
    :func:`scan_celebahq_tree`). Hashing and Pillow decoding are separate reads.
    This verifies the currently mounted bytes, not original archive provenance.
    """
    if checkpoint_every < 1 or progress_every < 1:
        raise ValueError("checkpoint_every and progress_every must be positive")

    try:
        from PIL import Image
    except ImportError as exc:  # pragma: no cover - Kaggle normally provides Pillow
        raise RuntimeError("Pillow is required for image decode verification") from exc

    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"DIRE root is not a directory: {root_path}")
    rows = list(records)
    if not rows:
        raise ValueError("no indexed CelebA-HQ records were supplied")
    work_path = Path(workdir).expanduser().resolve()
    if _inside(work_path, root_path):
        raise ValueError("checkpoint/output directory must be outside the read-only input dataset")

    fingerprint = _source_fingerprint(rows, root_path)
    run_id = fingerprint[:20]
    work_path.mkdir(parents=True, exist_ok=True)
    db_path = work_path / f"celebahq-integrity-{run_id}.sqlite3"
    manifest_path = work_path / f"celebahq-integrity-{run_id}.jsonl"
    summary_path = work_path / f"celebahq-integrity-{run_id}-summary.json"

    db = sqlite3.connect(db_path)
    try:
        _init_db(db, root=root_path, fingerprint=fingerprint, count=len(rows))
        db.commit()
        hashed_now = 0
        reused = 0
        pending = 0
        started = time.time()

        for index, record in enumerate(rows, start=1):
            path = Path(record.path)
            stat = path.stat()
            previous = db.execute(
                "SELECT size_bytes, mtime_ns, sha256, hash_status, decode_status "
                "FROM files WHERE relative_path = ?",
                (record.relative_path,),
            ).fetchone()
            if (
                previous
                and previous[0] == stat.st_size
                and previous[1] == stat.st_mtime_ns
                and previous[2]
                and previous[3] == "pass"
                and previous[4] in {"pass", "fail"}
            ):
                reused += 1
            else:
                hash_status = "pass"
                sha256 = None
                decode_status = "fail"
                image_format = None
                width = height = None
                mode = None
                errors: list[str] = []
                try:
                    sha256 = _sha256_file(path)
                except OSError as exc:
                    hash_status = "fail"
                    errors.append(f"sha256: {exc}")
                try:
                    with Image.open(path) as image:
                        image_format = image.format
                        width, height = image.size
                        mode = image.mode
                        image.load()  # force full pixel decode; do not retain pixel data
                    decode_status = "pass"
                except Exception as exc:  # record bad files and continue through the inventory
                    errors.append(f"decode: {type(exc).__name__}: {exc}")

                db.execute(
                    """INSERT OR REPLACE INTO files(
                        relative_path, split, generator, label, member_locator,
                        locator_inferred, size_bytes, mtime_ns, sha256, hash_status,
                        decode_status, image_format, width, height, mode, error
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                    (
                        record.relative_path,
                        record.split,
                        record.generator,
                        int(record.label),
                        record.member_locator,
                        int(record.locator_inferred),
                        stat.st_size,
                        stat.st_mtime_ns,
                        sha256,
                        hash_status,
                        decode_status,
                        image_format,
                        width,
                        height,
                        mode,
                        " | ".join(errors)[:1000] or None,
                    ),
                )
                hashed_now += 1
                pending += 1
                if pending >= checkpoint_every:
                    db.commit()
                    pending = 0

            if index % progress_every == 0 or index == len(rows):
                db.commit()
                print(
                    f"integrity {index}/{len(rows)}; hashed this run={hashed_now}; "
                    f"reused checkpoint={reused}; elapsed={time.time() - started:.1f}s"
                )

        db.commit()
        total = db.execute("SELECT COUNT(*) FROM files").fetchone()[0]
        total_file_bytes = db.execute("SELECT COALESCE(SUM(size_bytes), 0) FROM files").fetchone()[0]
        hash_failures = db.execute("SELECT COUNT(*) FROM files WHERE hash_status != 'pass'").fetchone()[0]
        decode_failures = db.execute("SELECT COUNT(*) FROM files WHERE decode_status != 'pass'").fetchone()[0]
        missing_hashes = db.execute("SELECT COUNT(*) FROM files WHERE sha256 IS NULL").fetchone()[0]
        group_counts = [
            {
                "split": split,
                "generator": generator,
                "label": label,
                "files": count,
                "hash_failures": hash_errors,
                "decode_failures": decode_errors,
            }
            for split, generator, label, count, hash_errors, decode_errors in db.execute(
                """SELECT split, generator, label, COUNT(*),
                          SUM(CASE WHEN hash_status != 'pass' THEN 1 ELSE 0 END),
                          SUM(CASE WHEN decode_status != 'pass' THEN 1 ELSE 0 END)
                   FROM files GROUP BY split, generator, label
                   ORDER BY split, generator"""
            )
        ]

        temporary_manifest = manifest_path.with_suffix(manifest_path.suffix + ".tmp")
        metadata = {
            "record_type": "metadata",
            "schema_version": 1,
            "root": str(root_path),
            "source_fingerprint": fingerprint,
            "source_fingerprint_basis": "relative path + stat size + mtime_ns; no content identity",
            "archive_member_paths_verified": False,
            "input_modified": False,
            "gpu_work_started": False,
            "records": total,
        }
        with temporary_manifest.open("w", encoding="utf-8") as stream:
            stream.write(json.dumps(metadata, ensure_ascii=False) + "\n")
            cursor = db.execute(
                """SELECT relative_path, split, generator, label, member_locator,
                          locator_inferred, size_bytes, sha256, hash_status,
                          decode_status, image_format, width, height, mode, error
                   FROM files ORDER BY split, generator, relative_path"""
            )
            for row in cursor:
                payload = dict(zip((
                    "path_from_dire_root", "split", "generator", "label",
                    "member_locator_inferred", "locator_inferred", "size_bytes",
                    "sha256", "hash_status", "decode_status", "image_format",
                    "width", "height", "mode", "error",
                ), row))
                payload["locator_inferred"] = bool(payload["locator_inferred"])
                stream.write(json.dumps(payload, ensure_ascii=False) + "\n")
        os.replace(temporary_manifest, manifest_path)

        summary: dict[str, Any] = {
            "schema_version": 1,
            "status": "pass" if total == len(rows) and not hash_failures and not decode_failures else "fail",
            "root": str(root_path),
            "source_fingerprint": fingerprint,
            "source_fingerprint_basis": "relative path + stat size + mtime_ns; no content identity",
            "input_modified": False,
            "image_contents_read": True,
            "gpu_work_started": False,
            "archive_member_paths_verified": False,
            "records_expected": len(rows),
            "records_in_checkpoint": total,
            "total_file_bytes_by_stat": total_file_bytes,
            "hashed_this_run": hashed_now,
            "reused_from_checkpoint": reused,
            "hash_failures": hash_failures,
            "missing_hashes": missing_hashes,
            "decode_failures": decode_failures,
            "groups": group_counts,
            "checkpoint_sqlite": str(db_path),
            "file_manifest_jsonl": str(manifest_path),
            "elapsed_seconds_this_run": round(time.time() - started, 2),
            "notes": [
                "SHA-256 covers the currently mounted file bytes only; it does not prove original archive or source provenance.",
                "Images are fully decoded one at a time after hashing; the input tree is never written.",
                "Checkpoint reuse assumes an immutable, identical mounted Dataset when path, size, and mtime match.",
            ],
        }
        temporary_summary = summary_path.with_suffix(summary_path.suffix + ".tmp")
        temporary_summary.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        os.replace(temporary_summary, summary_path)
        summary["summary_json"] = str(summary_path)
        return summary
    finally:
        db.close()


In [ ]:
# Stage 2: only after path-index cell succeeded; reuses `DIRE_ROOT` and `records`
integrity_result = verify_celebahq_records(
    root=DIRE_ROOT,
    records=records,
    workdir=Path('/kaggle/working'),
    checkpoint_every=250,
    progress_every=1000,
)
print('Integrity status:', integrity_result['status'])
print('Records:', integrity_result['records_in_checkpoint'], '/', integrity_result['records_expected'])
print('Hashed this run:', integrity_result['hashed_this_run'])
print('Reused checkpoint:', integrity_result['reused_from_checkpoint'])
print('SHA failures:', integrity_result['hash_failures'])
print('Decode failures:', integrity_result['decode_failures'])
print('SQLite checkpoint:', integrity_result['checkpoint_sqlite'])
print('JSONL manifest:', integrity_result['file_manifest_jsonl'])
print('Summary:', integrity_result['summary_json'])


## Stage 3 — deterministic validation IDs (after integrity pass)

Run only after Stage 2 reports `pass`, with zero SHA/decode failures. This reads its JSONL and summary, partitions **train/real** and **train/sdv2** only, and writes a candidate manifest under `/kaggle/working`; no image reads, GPU, or input writes. It uses the approved salt/rank rule and reports exact-content duplicates crossing the candidate split for review. Do not train until you review the artifact and complete the separate resource preflight.


In [ ]:
"""Create the approved deterministic CelebA-HQ train/validation ID split.

The splitter only reads the integrity summary/JSONL produced by Stage 2 and
writes a candidate split manifest beneath a caller-provided directory outside
the mounted input. It never opens images or changes the data tree.
"""

from __future__ import annotations

import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

VALIDATION_SALT = "dire-celebahq-val-v1"
EXPECTED_TRAIN_COUNTS = {("real", 0): 28_000, ("sdv2", 1): 40_000}
EXPECTED_TOTAL_RECORDS = 72_600


def _read_jsonl(path: Path) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    with path.open("r", encoding="utf-8") as stream:
        try:
            metadata = json.loads(next(stream))
        except StopIteration as exc:
            raise ValueError("integrity JSONL manifest is empty") from exc
        rows = [json.loads(line) for line in stream if line.strip()]
    if metadata.get("record_type") != "metadata":
        raise ValueError("integrity JSONL first line is not metadata")
    return metadata, rows


def _round_half_up_ratio(n: int, numerator: int, denominator: int) -> int:
    """Round ``n * numerator / denominator`` to nearest integer, ties upward."""
    return (2 * n * numerator + denominator) // (2 * denominator)


def _rank_digest(salt: str, locator: str) -> str:
    return hashlib.sha256(salt.encode("utf-8") + b"\0" + locator.encode("utf-8")).hexdigest()


def _canonical_list_sha256(values: list[Any]) -> str:
    payload = json.dumps(values, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def create_celebahq_validation_split(
    integrity_summary_path: str | Path,
    output_dir: str | Path,
    *,
    integrity_manifest_path: str | Path | None = None,
    salt: str = VALIDATION_SALT,
    expected_train_counts: dict[tuple[str, int], int] | None = None,
    expected_total_records: int | None = EXPECTED_TOTAL_RECORDS,
) -> dict[str, Any]:
    """Select exactly 10% of each training label using the locked hash-rank rule.

    The integrity summary and its JSONL manifest must describe the same complete,
    failure-free Stage 2 run. ``integrity_manifest_path`` can override the path
    embedded in the summary, allowing artifacts mounted read-only from a separate
    Kaggle Dataset. Only ``train/real`` and ``train/sdv2`` are split; all test
    records are excluded. All train rows sharing a SHA-256 with any other train
    row are ineligible for validation, globally across both labels; validation
    fills the exact per-label quota from the next unique-content rank candidates.
    """
    summary_path = Path(integrity_summary_path).expanduser().resolve()
    if not summary_path.is_file():
        raise FileNotFoundError(f"integrity summary not found: {summary_path}")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary.get("status") != "pass":
        raise ValueError("integrity summary status is not pass; refusing to create validation IDs")
    if summary.get("input_modified") is not False or summary.get("gpu_work_started") is not False:
        raise ValueError("integrity summary does not attest to read-only, no-GPU Stage 2")
    for key in ("hash_failures", "missing_hashes", "decode_failures"):
        if summary.get(key) != 0:
            raise ValueError(f"integrity summary has {key}={summary.get(key)!r}; refusing split")

    record_count = summary.get("records_in_checkpoint")
    if not isinstance(record_count, int) or record_count < 1:
        raise ValueError("integrity summary has an invalid records_in_checkpoint")
    if summary.get("records_expected") != record_count:
        raise ValueError("integrity summary expected/checkpoint record counts differ")
    if expected_total_records is not None and record_count != expected_total_records:
        raise ValueError(
            f"expected {expected_total_records} integrity records, got {record_count}"
        )

    manifest_value = integrity_manifest_path or summary.get("file_manifest_jsonl")
    if not isinstance(manifest_value, (str, Path)) or not str(manifest_value):
        raise ValueError("integrity summary does not name file_manifest_jsonl")
    manifest_path = Path(manifest_value).expanduser().resolve()
    if not manifest_path.is_file():
        raise FileNotFoundError(f"integrity JSONL manifest not found: {manifest_path}")
    manifest_meta, records = _read_jsonl(manifest_path)
    if len(records) != record_count or manifest_meta.get("records") != record_count:
        raise ValueError("integrity JSONL record count does not match its summary")
    if manifest_meta.get("source_fingerprint") != summary.get("source_fingerprint"):
        raise ValueError("integrity JSONL and summary source fingerprints differ")
    if manifest_meta.get("archive_member_paths_verified") is not False:
        raise ValueError("unexpected archive-member path verification claim")

    seen_paths: set[str] = set()
    classes: dict[int, list[dict[str, Any]]] = defaultdict(list)
    observed_train_counts: Counter[tuple[str, int]] = Counter()
    for record in records:
        if record.get("hash_status") != "pass" or record.get("decode_status") != "pass":
            raise ValueError(f"integrity failure in manifest row: {record.get('path_from_dire_root')}")
        image_hash = record.get("sha256")
        if not isinstance(image_hash, str) or len(image_hash) != 64:
            raise ValueError(f"missing/invalid SHA-256 for {record.get('path_from_dire_root')}")
        try:
            int(image_hash, 16)
        except ValueError as exc:
            raise ValueError(f"invalid SHA-256 for {record.get('path_from_dire_root')}") from exc
        path_id = record.get("path_from_dire_root")
        if not isinstance(path_id, str) or not path_id or path_id in seen_paths:
            raise ValueError(f"missing or duplicate root-relative path: {path_id!r}")
        seen_paths.add(path_id)

        if record.get("split") != "train":
            continue
        generator = record.get("generator")
        label = record.get("label")
        if (generator, label) not in EXPECTED_TRAIN_COUNTS:
            raise ValueError(f"unexpected CelebA-HQ training group: {(generator, label)!r}")
        locator = record.get("member_locator_inferred")
        if not isinstance(locator, str) or not locator:
            raise ValueError(f"missing inferred member locator for {path_id}")
        if record.get("locator_inferred") is not True:
            raise ValueError(f"expected locator_inferred=true for {path_id}")
        observed_train_counts[(generator, label)] += 1
        classes[label].append(record)

    required_counts = expected_train_counts or EXPECTED_TRAIN_COUNTS
    if dict(observed_train_counts) != required_counts:
        raise ValueError(
            f"training group counts differ from locked inventory: "
            f"observed={dict(observed_train_counts)}, expected={required_counts}"
        )
    if set(classes) != {0, 1}:
        raise ValueError(f"expected binary train labels 0 and 1, got {sorted(classes)}")
    for label, class_records in classes.items():
        locators = [row["member_locator_inferred"] for row in class_records]
        if len(locators) != len(set(locators)):
            raise ValueError(f"duplicate inferred member locator within label {label}")

    all_train_records = [record for class_records in classes.values() for record in class_records]
    global_sha_counts = Counter(record["sha256"] for record in all_train_records)
    validation_records: list[dict[str, Any]] = []
    class_counts: dict[str, dict[str, int]] = {}
    candidate_audit_by_label: dict[str, dict[str, int]] = {}
    validation_path_ids: set[str] = set()
    for label in sorted(classes):
        ranked = sorted(
            classes[label],
            key=lambda row: (
                _rank_digest(salt, row["member_locator_inferred"]),
                row["member_locator_inferred"],
            ),
        )
        selected_count = _round_half_up_ratio(len(ranked), 1, 10)
        eligible = [
            (rank_position, row)
            for rank_position, row in enumerate(ranked, start=1)
            if global_sha_counts[row["sha256"]] == 1
        ]
        excluded_duplicate_candidates = len(ranked) - len(eligible)
        if len(eligible) < selected_count:
            raise ValueError(
                f"not enough content-unique candidates for label {label}: "
                f"need {selected_count}, have {len(eligible)}"
            )
        chosen = eligible[:selected_count]
        for validation_rank, (rank_position, row) in enumerate(chosen, start=1):
            path_id = row["path_from_dire_root"]
            validation_path_ids.add(path_id)
            validation_records.append(
                {
                    "path_from_dire_root": path_id,
                    "generator": row["generator"],
                    "label": label,
                    "member_locator_inferred": row["member_locator_inferred"],
                    "locator_inferred": True,
                    "sha256": row["sha256"],
                    "validation_rank": validation_rank,
                    "rank_position_before_duplicate_filter": rank_position,
                    "selection_sha256": _rank_digest(salt, row["member_locator_inferred"]),
                }
            )
        class_counts[str(label)] = {
            "total": len(ranked),
            "validation": selected_count,
            "training": len(ranked) - selected_count,
        }
        candidate_audit_by_label[str(label)] = {
            "total_records": len(ranked),
            "content_unique_candidates": len(eligible),
            "excluded_duplicate_hash_candidates": excluded_duplicate_candidates,
        }

    validation_records.sort(key=lambda row: (row["label"], row["validation_rank"]))
    validation_id_list = sorted(
        [
            {
                "label": row["label"],
                "path_from_dire_root": row["path_from_dire_root"],
                "member_locator_inferred": row["member_locator_inferred"],
            }
            for row in validation_records
        ],
        key=lambda item: (item["label"], item["member_locator_inferred"], item["path_from_dire_root"]),
    )
    validation_hashes_all = {
        row["sha256"] for row in all_train_records
        if row["path_from_dire_root"] in validation_path_ids
    }
    training_hashes_all = {
        row["sha256"] for row in all_train_records
        if row["path_from_dire_root"] not in validation_path_ids
    }
    shared_hashes_all = sorted(validation_hashes_all & training_hashes_all)
    shared_hashes: dict[str, list[str]] = {}
    for label in sorted(classes):
        validation_hashes = {
            row["sha256"] for row in classes[label]
            if row["path_from_dire_root"] in validation_path_ids
        }
        training_hashes = {
            row["sha256"] for row in classes[label]
            if row["path_from_dire_root"] not in validation_path_ids
        }
        shared_hashes[str(label)] = sorted(validation_hashes & training_hashes)

    content_overlap = {
        "status": "review_required" if shared_hashes_all else "pass",
        "shared_sha256_count_all_labels": len(shared_hashes_all),
        "shared_sha256_all_labels": shared_hashes_all,
        "shared_sha256_count_by_label": {label: len(values) for label, values in shared_hashes.items()},
        "shared_sha256_by_label": shared_hashes,
        "note": "Exact duplicate bytes crossing train/validation are reported across both labels; the locked rank selection is not silently changed.",
    }
    output_path = Path(output_dir).expanduser().resolve()
    root_path = Path(summary.get("root", "")).expanduser().resolve()
    if not summary.get("root") or output_path == root_path or root_path in output_path.parents:
        raise ValueError("validation manifest output must be outside the read-only input root")
    output_path.mkdir(parents=True, exist_ok=True)
    validation_ids_digest = _canonical_list_sha256(validation_id_list)
    artifact = {
        "schema_version": 2,
        "status": "candidate_created",
        "protocol": {
            "name": "celebahq-content-disjoint-exact-count-hash-rank-v2",
            "salt": salt,
            "hash": "SHA-256",
            "preimage": "UTF-8(salt) + 0x00 + UTF-8(member_locator_inferred)",
            "sort": "ascending lowercase hex digest, then member_locator_inferred",
            "candidate_rule": "candidate SHA-256 frequency must equal 1 across all train labels; repeated-content groups remain wholly in training",
            "fraction": {"numerator": 1, "denominator": 10},
            "rounding": "half-up of total records per label before duplicate filtering",
        },
        "source": {
            "root": summary["root"],
            "source_fingerprint": summary["source_fingerprint"],
            "integrity_summary": str(summary_path),
            "integrity_manifest": str(manifest_path),
            "integrity_status": summary["status"],
            "records": record_count,
            "archive_member_paths_verified": False,
        },
        "counts_by_label": class_counts,
        "candidate_audit_by_label": candidate_audit_by_label,
        "duplicate_sha256_groups_all_train_labels": sum(
            1 for frequency in global_sha_counts.values() if frequency > 1
        ),
        "validation_id_list_sha256": validation_ids_digest,
        "content_overlap_audit": content_overlap,
        "validation_records": validation_records,
        "notes": [
            "Only train/real and train/sdv2 records are partitioned; all test records remain untouched.",
            "Repeated SHA-256 groups across either training label are excluded from validation candidates and remain entirely in training.",
            "Locators are inferred from the extracted tree and have not been verified against original tar member paths.",
            "Candidate split creation does not authorize training; review content_overlap_audit and complete the compute preflight first.",
        ],
    }
    destination = output_path / f"celebahq-validation-split-v2-{summary['source_fingerprint'][:12]}.json"
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.write_text(json.dumps(artifact, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, destination)
    artifact["artifact_path"] = str(destination)
    return artifact


In [ ]:
# Stage 3 runner — use the just-finished Stage 2 result in this runtime when available.
from pathlib import Path

if isinstance(globals().get("integrity_result"), dict) and integrity_result.get("summary_json"):
    _integrity_summary_path = Path(integrity_result["summary_json"])
else:
    # Fallback to the concrete successful Stage 2 output reported for this run.
    _integrity_summary_path = Path(
        "/kaggle/working/celebahq-integrity-81dcda99aeb997b25b82-summary.json"
    )

validation_result = create_celebahq_validation_split(
    integrity_summary_path=_integrity_summary_path,
    output_dir=Path("/kaggle/working"),
)
print("Validation candidate:", validation_result["status"])
print("Counts by label:", validation_result["counts_by_label"])
print("Candidate audit by label:", validation_result["candidate_audit_by_label"])
print("Validation ID-list SHA-256:", validation_result["validation_id_list_sha256"])
print("Content-overlap audit:", validation_result["content_overlap_audit"]["status"])
print("Split artifact:", validation_result["artifact_path"])
